# 03 — Compare eval runs

Globs `data/eval_results/retrieval_*.json`, joins on `(model, dims, chunker)`, plots per-source metric bars, highlights the winners.

Use this after running `eval-retrieval` across N candidate configs in Phase C — it's the comparison view the CLI doesn't yet emit on its own.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS_DIR = Path("data/eval_results")

In [ ]:
# Load all retrieval_*.json into a flat DataFrame keyed by (run, source).
rows = []
for path in sorted(RESULTS_DIR.glob("retrieval_*.json")):
    data = json.loads(path.read_text())
    config_label = (
        f"{data['embedding_model']}@{data['embedding_dims']}"
        f" | {','.join(f'{k}={v}' for k, v in data['chunker_by_source'].items())}"
    )
    for m in data["per_source"]:
        rows.append(
            {
                "run": path.stem,
                "config": config_label,
                "started_at": data["started_at"],
                **m,
            }
        )
df = pd.DataFrame(rows)
df

In [ ]:
# Per-source winner on Recall@5.
for source, group in df.groupby("source"):
    best = group.sort_values("recall_at_5", ascending=False).iloc[0]
    print(f"{source:14s} winner: {best['config']}  recall@5={best['recall_at_5']:.3f}")

In [ ]:
# Side-by-side bars — per-source Recall@5 per config.
pivot = df.pivot_table(
    index="source", columns="config", values="recall_at_5"
)
pivot.plot.bar(figsize=(10, 5), ylim=(0, 1))
plt.title("Recall@5 per source per config")
plt.ylabel("Recall@5")
plt.xticks(rotation=0)
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

In [ ]:
# Same shape for MRR@10 and nDCG@10 if needed.
for metric in ("mrr_at_10", "ndcg_at_10"):
    df.pivot_table(index="source", columns="config", values=metric).plot.bar(
        figsize=(10, 5), ylim=(0, 1)
    )
    plt.title(f"{metric} per source per config")
    plt.xticks(rotation=0)
    plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.show()